# Week 2: Row Serialization & Prompt Engineering
**Goal:** Build and test the pipeline that converts tabular rows into prompts Nemotron can classify.

**Depends on:** `week1_baselines.ipynb` must be run first to generate the shared splits.

**This notebook:**
1. Loads a sample from the Week 1 train split for manual testing
2. Builds `serialize_row()`: converts a row to a natural language sentence
3. Builds zero-shot and few-shot prompt functions
4. Builds `parse_response()`: extracts label and reasoning trace
5. Tests manually on 10 rows from the train set
6. Saves finalized functions to `src/`

> **Note:** Week 2 uses the **train set** for manual testing only.
> The **test set** is reserved exclusively for Week 3 experiments.

## 1. Imports

In [1]:
import pandas as pd
import requests
import json
import time
import re
import os

print("Imports OK")

Imports OK


## 2. Load Sample from Week 1 Train Split

Load a small sample from the train set for manual testing.
We use the train set here — the test set is reserved for Week 3.

> Run `week1_baselines.ipynb` first to generate `data/week3_train_5000.csv`.

In [2]:
import os

TRAIN_PATH = "../data/week3_train_5000.csv"

if not os.path.exists(TRAIN_PATH):
    print("⚠ Train split not found.")
    print("  Run week1_baselines.ipynb first to generate the shared splits.")
else:
    df_train = pd.read_csv(TRAIN_PATH)

    # Filter uninformative occupations
    df_train = df_train[
        ~df_train["occupation"].str.lower().str.strip().str.replace("_", " ").isin([
            "no occupation", "not in workforce", ""
        ]) &
        (df_train["occupation"].str.strip() != "") &
        (df_train["occupation"].notna())
    ].reset_index(drop=True)

    # Use a 50-row sample for manual testing
    sample = df_train.sample(50, random_state=42).reset_index(drop=True)

    print(f"Train split loaded: {len(df_train):,} rows")
    print(f"Manual test sample: {len(sample)} rows")
    print(f"Label balance:")
    print(sample["label_name"].value_counts())
    print()
    sample.head(3)


Train split loaded: 2,460 rows
Manual test sample: 50 rows
Label balance:
label_name
not_college    33
college        17
Name: count, dtype: int64



## 3. Row Serialization

This converts one Nemotron-Personas-USA row into a natural language string that Nemotron can reason about.
The occupation column is the most semantically rich feature. We will keep the full name. 

In [3]:
def serialize_row(row: dict) -> str:
    
    occupation = str(row["occupation"]).replace("_", " ").strip()
    marital    = str(row["marital_status"]).replace("_", " ").strip()
    sex        = str(row["sex"]).lower().strip()
    state      = str(row["state"]).strip()
    age        = int(row["age"])

    return (
        f"A {age}-year-old {sex}, {marital}, "
        f"working as a {occupation}. "
        f"Located in {state}."
    )

# Test on 5 rows
print("Serialization examples")
print()
for _, row in sample.sample(5, random_state=7).iterrows():
    print(f"Input:  {serialize_row(row.to_dict())}")
    print(f"Label:  {row['label_name']} ({row['education_level']})")
    print()

Serialization examples

Input:  A 47-year-old male, divorced, working as a laborer or freight stock or material mover. Located in TN.
Label:  not_college (high_school)

Input:  A 60-year-old female, married present, working as a customer service representative. Located in NV.
Label:  college (bachelors)

Input:  A 39-year-old female, married present, working as a project management specialist. Located in NY.
Label:  college (graduate)

Input:  A 53-year-old female, widowed, working as a customer service representative. Located in LA.
Label:  not_college (high_school)

Input:  A 23-year-old female, married present, working as a security guard or gambling surveillance officer. Located in MS.
Label:  not_college (high_school)



## 4. Prompt Builders

In [4]:
SYSTEM_PROMPT = """You are an education level classifier. Given a person's demographic information,
predict whether they are college-educated (have a bachelor's degree or higher) or not.
Think step by step, then respond with ONLY one of these two labels on the final line: college or not_college."""


def build_zero_shot_prompt(row: dict) -> list:
    # Build a zero-shot chat prompt for one row
    description = serialize_row(row)
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": (
            f"Classify this person's education level:\n\n"
            f"{description}\n\n"
            f"Answer with college or not_college only."
        )}
    ]


def build_few_shot_prompt(row: dict, examples: list) -> list:

    # Build a few-shot chat prompt for one row.
    # examples: list of dicts with keys 'row' (dict) and 'label' (str)

    example_text = ""
    for ex in examples:
        example_text += f"Person: {serialize_row(ex['row'])}\nLabel: {ex['label']}\n\n"

    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": (
            f"Here are some labeled examples:\n\n{example_text}"
            f"Now classify this person:\n\n"
            f"{serialize_row(row)}\n\n"
            f"Answer with college or not_college only."
        )}
    ]


# Preview both prompt types on one row
test_row = sample.iloc[0].to_dict()

print("=== ZERO-SHOT PROMPT ===")
zs = build_zero_shot_prompt(test_row)
for msg in zs:
    print(f"[{msg['role'].upper()}]\n{msg['content']}\n")

print("\n=== FEW-SHOT PROMPT (3 examples) ===")
few_shot_pool = sample[sample["label_name"] == "college"].head(2).to_dict("records") + \
                sample[sample["label_name"] == "not_college"].head(1).to_dict("records")
examples = [{"row": r, "label": r["label_name"]} for r in few_shot_pool]
fs = build_few_shot_prompt(test_row, examples)
for msg in fs:
    print(f"[{msg['role'].upper()}]\n{msg['content']}\n")

=== ZERO-SHOT PROMPT ===
[SYSTEM]
You are an education level classifier. Given a person's demographic information,
predict whether they are college-educated (have a bachelor's degree or higher) or not.
Think step by step, then respond with ONLY one of these two labels on the final line: college or not_college.

[USER]
Classify this person's education level:

A 37-year-old male, never married, working as a manager. Located in FL.

Answer with college or not_college only.


=== FEW-SHOT PROMPT (3 examples) ===
[SYSTEM]
You are an education level classifier. Given a person's demographic information,
predict whether they are college-educated (have a bachelor's degree or higher) or not.
Think step by step, then respond with ONLY one of these two labels on the final line: college or not_college.

[USER]
Here are some labeled examples:

Person: A 37-year-old male, never married, working as a manager. Located in FL.
Label: college

Person: A 29-year-old male, never married, working as a comput

## 5. Response Parser

Nemotron 3 Nano wraps its reasoning in `<think>...</think>` tags.
We extract the reasoning trace and the final label separately.

In [5]:
def parse_response(content: str) -> tuple:
    """
    Extract label and reasoning trace from Nemotron's response.
    Returns: (label, trace) where label is 'college', 'not_college', or 'UNKNOWN'
    """
    # Extract <think>...</think> reasoning trace
    think_match = re.search(r"<think>(.*?)</think>", content, re.DOTALL)
    trace = think_match.group(1).strip() if think_match else ""

    # Search for label where not_college must come before college in the regex
    # to avoid matching 'college' inside 'not_college'
    label_match = re.search(r"\b(not_college|college)\b", content, re.IGNORECASE)
    label = label_match.group(1).lower() if label_match else "UNKNOWN"

    return label, trace


# Test the parser on mock responses
mock_responses = [
    "<think>This person works as a surgeon which requires medical school.\nThey are highly educated.</think>\ncollege",
    "<think>Fast food worker typically does not require a college degree.</think>\nnot_college",
    "Based on the information provided, I would classify this as: not_college",
    "I cannot determine the answer.",  # UNKNOWN case
]

print("=== Parser tests ===")
for resp in mock_responses:
    label, trace = parse_response(resp)
    print(f"Label: {label:12s} | Trace: {trace[:60]}..." if trace else f"Label: {label:12s} | No trace")

=== Parser tests ===
Label: college      | Trace: This person works as a surgeon which requires medical school...
Label: college      | Trace: Fast food worker typically does not require a college degree...
Label: not_college  | No trace
Label: UNKNOWN      | No trace


## 6. Ollama Setup Check

Before running inference, verify Ollama is installed and the Nemotron model is available.

> **If Ollama is not installed yet:** Download from https://ollama.com and install it.
> Then in a terminal run: `ollama pull nemotron-mini` or `ollama pull hf.co/nvidia/NVIDIA-Nemotron-3-Nano-4B-GGUF`
> Keep Ollama running in the background while using this notebook.

In [6]:
OLLAMA_URL = "http://localhost:11434/v1/chat/completions"
MODEL_NAME = "nemotron-3-nano:4b"  # confirmed working model tag

def check_ollama():
    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=5)
        models = [m["name"] for m in r.json().get("models", [])]
        print(f"Ollama is running. Available models: {models}")
        if MODEL_NAME in models:
            print(f"✓ {MODEL_NAME} is ready")
        else:
            print(f"⚠ {MODEL_NAME} not found — run: ollama pull nemotron-3-nano:4b")
        return True
    except Exception as e:
        print(f"Ollama not reachable: {e}")
        print("Make sure Ollama is installed and running (ollama serve)")
        return False

ollama_ready = check_ollama()


Ollama is running. Available models: ['nemotron-3-nano:4b', 'nemotron-3-nano:30b-a3b-q4_K_M']
✓ nemotron-3-nano:4b is ready


## 7. Inference Function

In [7]:
def classify_row(messages: list, enable_thinking: bool = True, timeout: int = 120) -> dict:

    # Send a prompt to Ollama and return label, reasoning trace, timing, and token count.

    payload = {
        "model": MODEL_NAME,
        "messages": messages,
        "max_tokens": 1024,
        "temperature": 0.1,   # low temperature for more consistent labels
        "stream": False,
    }

    # Nemotron thinking toggle  append /no_think to disable reasoning)
    if not enable_thinking:
        payload["messages"] = [
            {"role": m["role"], "content": m["content"] + (" /no_think" if m["role"] == "user" else "")}
            for m in messages
        ]

    start = time.time()
    try:
        response = requests.post(OLLAMA_URL, json=payload, timeout=timeout).json()
        elapsed = time.time() - start
        content = response["choices"][0]["message"]["content"]
        tokens  = response.get("usage", {}).get("completion_tokens", 0)
        label, trace = parse_response(content)
    except Exception as e:
        elapsed = time.time() - start
        content, label, trace, tokens = str(e), "ERROR", "", 0

    return {
        "label":   label,
        "trace":   trace,
        "raw":     content,
        "time_ms": round(elapsed * 1000),
        "tokens":  tokens,
    }

print("classify_row() defined.")

classify_row() defined.


## 8. Manual Test: 10 Rows from Train Set

Run zero-shot classification on 10 rows from the **train set**.
This is a sanity check — the test set is reserved for Week 3.

> Skip this cell if Ollama is not running yet.

In [8]:
if not ollama_ready:
    print("Skipping — Ollama not running.")
else:
    test_10 = sample.sample(10, random_state=7).reset_index(drop=True)
    manual_results = []

    for i, row in test_10.iterrows():
        row_dict   = row.to_dict()
        messages   = build_zero_shot_prompt(row_dict)
        result     = classify_row(messages, enable_thinking=True)
        true_label = row["label_name"]
        correct    = result["label"] == true_label

        print(f"Row {i+1:2d} | Input: {serialize_row(row_dict)}")
        print(f"       | True: {true_label:12s} | Predicted: {result['label']:12s} | {'✓' if correct else '✗'} | {result['time_ms']}ms")
        if result["trace"]:
            print(f"       | Reasoning: {result['trace'][:120]}...")
        print()

        manual_results.append({
            "input":      serialize_row(row_dict),
            "true_label": true_label,
            "pred_label": result["label"],
            "correct":    correct,
            "time_ms":    result["time_ms"],
            "tokens":     result["tokens"],
            "trace":      result["trace"],
            "raw":        result["raw"],
        })

    results_df = pd.DataFrame(manual_results)
    accuracy   = results_df["correct"].mean()
    print(f"Manual test accuracy (10 rows from train set): {accuracy:.0%}")
    print("Note: This is just a pipeline check — real results come from Week 3 test set.")

    os.makedirs("../results", exist_ok=True)
    results_df.to_csv("../results/week2_manual_tests.csv", index=False)
    print("Saved: results/week2_manual_tests.csv")


Row  1 | Input: A 47-year-old male, divorced, working as a laborer or freight stock or material mover. Located in TN.
       | True: not_college  | Predicted: college      | ✗ | 3490ms

Row  2 | Input: A 60-year-old female, married present, working as a customer service representative. Located in NV.
       | True: college      | Predicted: not_college  | ✗ | 3265ms

Row  3 | Input: A 39-year-old female, married present, working as a project management specialist. Located in NY.
       | True: college      | Predicted: not_college  | ✗ | 3693ms

Row  4 | Input: A 53-year-old female, widowed, working as a customer service representative. Located in LA.
       | True: not_college  | Predicted: not_college  | ✓ | 3439ms

Row  5 | Input: A 23-year-old female, married present, working as a security guard or gambling surveillance officer. Located in MS.
       | True: not_college  | Predicted: college      | ✗ | 3633ms

Row  6 | Input: A 31-year-old female, married present, working as a stoc

## 9. Inspect Best and Worst Reasoning Traces

Look at which rows the model got right vs wrong and why.

In [9]:
try:
    results_df
except NameError:
    results_df = pd.read_csv("../results/week2_manual_tests.csv")

print("=== CORRECT predictions — reasoning traces ===")
for _, row in results_df[results_df["correct"]].head(2).iterrows():
    print(f"Input:     {row['input']}")
    print(f"Label:     {row['true_label']}")
    print(f"Reasoning: {row['trace']}")
    print()

print("\n=== WRONG predictions — reasoning traces ===")
for _, row in results_df[~results_df["correct"]].head(2).iterrows():
    print(f"Input:     {row['input']}")
    print(f"True:      {row['true_label']} | Predicted: {row['pred_label']}")
    print(f"Reasoning: {row['trace']}")
    print()

=== CORRECT predictions — reasoning traces ===
Input:     A 53-year-old female, widowed, working as a customer service representative. Located in LA.
Label:     not_college
Reasoning: 

Input:     A 31-year-old female, married present, working as a stocker or order filler. Located in WA.
Label:     not_college
Reasoning: 


=== WRONG predictions — reasoning traces ===
Input:     A 47-year-old male, divorced, working as a laborer or freight stock or material mover. Located in TN.
True:      not_college | Predicted: college
Reasoning: 

Input:     A 60-year-old female, married present, working as a customer service representative. Located in NV.
True:      college | Predicted: not_college
Reasoning: 



## 10. Save Finalized Functions to src/

Write the final versions to `src/` Python files.
These are imported by Week 3 for the full experiments.

In [10]:
serialize_code = '''
def serialize_row(row: dict) -> str:
    """
    Convert one Nemotron-Personas-USA row into a natural language description.
    """
    occupation = str(row["occupation"]).replace("_", " ").strip()
    marital    = str(row["marital_status"]).replace("_", " ").strip()
    sex        = str(row["sex"]).lower().strip()
    state      = str(row["state"]).strip()
    age        = int(row["age"])
    return (
        f"A {age}-year-old {sex}, {marital}, "
        f"working as a {occupation}. "
        f"Located in {state}."
    )
'''

prompts_code = '''
from src.serialize import serialize_row

SYSTEM_PROMPT = """You are an education level classifier. Given a person\'s demographic information,
predict whether they are college-educated (have a bachelor\'s degree or higher) or not.
Think step by step, then respond with ONLY one of these two labels on the final line: college or not_college."""


def build_zero_shot_prompt(row: dict) -> list:
    description = serialize_row(row)
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": (
            f"Classify this person\'s education level:\\n\\n"
            f"{description}\\n\\n"
            f"Answer with college or not_college only."
        )}
    ]


def build_few_shot_prompt(row: dict, examples: list) -> list:
    example_text = ""
    for ex in examples:
        example_text += f"Person: {serialize_row(ex[\'row\'])}\\nLabel: {ex[\'label\']}\\n\\n"
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": (
            f"Here are some labeled examples:\\n\\n{example_text}"
            f"Now classify this person:\\n\\n"
            f"{serialize_row(row)}\\n\\n"
            f"Answer with college or not_college only."
        )}
    ]
'''

parse_code = '''
import re

def parse_response(content: str) -> tuple:
    """
    Extract label and reasoning trace from Nemotron\'s response.
    Returns: (label, trace)
    """
    think_match = re.search(r"<think>(.*?)</think>", content, re.DOTALL)
    trace = think_match.group(1).strip() if think_match else ""
    label_match = re.search(r"\\b(not_college|college)\\b", content, re.IGNORECASE)
    label = label_match.group(1).lower() if label_match else "UNKNOWN"
    return label, trace
'''

src_dir = "../src"
os.makedirs(src_dir, exist_ok=True)
with open(f"{src_dir}/serialize.py", "w") as f:
    f.write(serialize_code.strip())
with open(f"{src_dir}/prompts.py", "w") as f:
    f.write(prompts_code.strip())
with open(f"{src_dir}/parse_response.py", "w") as f:
    f.write(parse_code.strip())

print("Saved:")
print("  src/serialize.py")
print("  src/prompts.py")
print("  src/parse_response.py")
print()
print("Week 2 complete — proceed to week3_v2.ipynb")


Saved:
  src/serialize.py
  src/prompts.py
  src/parse_response.py

Week 2 complete — proceed to week3_v2.ipynb


## Week 2 Summary

| Item | Status |
|---|---|
| `serialize_row()` written and tested | |
| `build_zero_shot_prompt()` working | |
| `build_few_shot_prompt()` working | |
| `parse_response()` tested | |
| Ollama running with `nemotron-3-nano:4b` | |
| 10-row manual test complete (train set) | |
| `src/` files saved | |

**Manual test accuracy (10 rows from train):** 60%

> This is a pipeline check only — real accuracy results come from Week 3.